# Notebook 04 - Deterministic Checks v1 POC


In [ ]:

# ============================================================
# Notebook 04 - Deterministic Checks
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import datetime as dt
import importlib.util
import types
from pathlib import Path

import yaml
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
RULES_PATH = f"{BASE_FILES_PATH}/rules"
AGENTS_YAML_PATH = f"{CONFIG_PATH}/agents.yaml"
DETERMINISTIC_RULES_PATH = f"{RULES_PATH}/deterministic_rules.py"

CURRENT_RUN_CASES_TABLE = "agent_eval_current_run_cases"
RESPONSES_TABLE = "agent_eval_agent_responses_staging"
DETERMINISTIC_RESULTS_TABLE = "agent_eval_deterministic_results"


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def is_onelake_path(path):
    return str(path).startswith("abfss://")


def file_exists(path):
    if is_onelake_path(path):
        return mssparkutils.fs.exists(path)
    return Path(path).is_file()


def read_text(path, max_bytes=20 * 1024 * 1024):
    if is_onelake_path(path):
        return mssparkutils.fs.head(path, max_bytes)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_agents():
    return {a["agent_id"]: a for a in (yaml.safe_load(read_text(AGENTS_YAML_PATH)) or {}).get("agents", [])}


def load_rules_module():
    if not file_exists(DETERMINISTIC_RULES_PATH):
        return None
    if is_onelake_path(DETERMINISTIC_RULES_PATH):
        module = types.ModuleType("deterministic_rules")
        module.__file__ = DETERMINISTIC_RULES_PATH
        exec(read_text(DETERMINISTIC_RULES_PATH), module.__dict__)
        return module
    spec = importlib.util.spec_from_file_location("deterministic_rules", DETERMINISTIC_RULES_PATH)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def make_result(test_id, agent_id, rule_name, passed, severity, evidence):
    return {
        "run_id": run_id,
        "test_id": test_id,
        "agent_id": agent_id,
        "rule_name": rule_name,
        "passed": bool(passed),
        "severity": severity,
        "evidence": evidence,
        "checked_at": now_utc(),
    }


def csv_rules(case, response):
    rows = []
    lower = (response or "").lower()
    must_contain = [p.strip() for p in (case.get("must_contain") or "").split("|") if p.strip()]
    must_not = [p.strip() for p in (case.get("must_not_contain") or "").split("|") if p.strip()]
    if must_contain:
        found = [p for p in must_contain if p.lower() in lower]
        rows.append(make_result(case["test_id"], case["agent_id"], "csv_must_contain_any", len(found) >= 1, "major", f"found_any={found}; alternatives={must_contain}"))
    if must_not:
        found = [p for p in must_not if p.lower() in lower]
        rows.append(make_result(case["test_id"], case["agent_id"], "csv_must_not_contain", len(found) == 0, "critical", f"forbidden_found={found}" if found else "No forbidden terms found"))
    return rows


def named_rules(case, response, agent, module):
    rows = []
    for rule_name in agent.get("deterministic_rules", []) or []:
        if module is None or not hasattr(module, rule_name):
            rows.append(make_result(case["test_id"], case["agent_id"], rule_name, False, "critical", "Rule not found in deterministic_rules.py"))
            continue
        try:
            result = getattr(module, rule_name)(response, {"case": case, "agent": agent})
            rows.append(make_result(case["test_id"], case["agent_id"], result.get("rule", rule_name), result.get("passed", False), result.get("severity", "major"), result.get("evidence", "")))
        except Exception as exc:
            rows.append(make_result(case["test_id"], case["agent_id"], rule_name, False, "critical", f"Rule error: {str(exc)[:300]}"))
    return rows


schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("test_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("severity", StringType(), False),
    StructField("evidence", StringType(), True),
    StructField("checked_at", TimestampType(), False),
])

agents = load_agents()
module = load_rules_module()
cases = {r["test_id"]: r.asDict() for r in spark.table(CURRENT_RUN_CASES_TABLE).filter(F.col("run_id") == run_id).collect()}
responses = [r.asDict() for r in spark.table(RESPONSES_TABLE).filter(F.col("run_id") == run_id).collect()]

rows = []
for response_row in responses:
    case = cases.get(response_row["test_id"])
    if not case:
        continue
    agent = agents.get(case["agent_id"], {})
    response = response_row.get("agent_response") or ""
    case_rows = []
    case_rows.extend(csv_rules(case, response))
    case_rows.extend(named_rules(case, response, agent, module))
    if not case_rows:
        case_rows.append(make_result(case["test_id"], case["agent_id"], "no_rules", True, "info", "No deterministic rules declared"))
    rows.extend(case_rows)

spark.createDataFrame([Row(**r) for r in rows], schema=schema).write.format("delta").mode("append").saveAsTable(DETERMINISTIC_RESULTS_TABLE)
critical_failures = [r for r in rows if (not r["passed"]) and r["severity"] == "critical"]
print(f"Deterministic checks complete. rows={len(rows)} critical_failures={len(critical_failures)}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
